# Instacart Market Basket -> Quantity Reconstruction Dataset

이 노트북은 `sample_data/insta_market_basket` 원자료를 TitanTPP quantity reconstruction 실험에 쓸 수 있는 형태로 재구성한다.

핵심 아이디어는 다음과 같다.

- `entity_id`: `user_id`
- `timestamp`: 유저별 `days_since_prior_order` 누적값으로 만든 상대 시간축
- `event row`: `order_products__prior.csv`, `order_products__train.csv`의 각 상품 구매 row
- `demand_qty`: `user_id + relative day bucket`별 상품 구매 row 수
- `mark`: 해당 active day basket에서 가장 많이 등장한 `department_id`

주의: Instacart 데이터에는 실제 calendar date가 없다. 따라서 이 데이터는 실제 시장 날짜별 수요가 아니라 **user-relative derived basket-count demand**로 해석해야 한다.

## 1. Imports and paths

노트북 실행 위치가 달라도 동작하도록 project root를 탐색한다. 출력 파일은 기본적으로 `sample_data/insta_market_basket` 아래에 저장한다.

In [1]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

from pathlib import Path
import sys

SERVER_PROJECT_ROOT = Path('~/workspace/paper_research').expanduser().resolve()
if str(SERVER_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVER_PROJECT_ROOT))

from utils.notebook_bootstrap import bootstrap_notebook

PROJECT_ROOT, DIR = bootstrap_notebook(preferred_root=SERVER_PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "sample_data" / "insta_market_basket"
OUTPUT_DIR = DATA_DIR

ORDER_PRODUCT_FILES = [
    "order_products__prior.csv",
    "order_products__train.csv",
]

CHUNKSIZE = 5_000_000
LOG_EVERY_N_CHUNKS = 1

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT = /home/leekwanhyeong/workspace/paper_research
PROJECT_ROOT: /home/leekwanhyeong/workspace/paper_research
DATA_DIR: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket
OUTPUT_DIR: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket


## 2. Load metadata tables

`products.csv`를 통해 상품을 aisle/department로 연결한다. 첫 실험에서는 mark class 수가 작고 안정적인 `department_id`를 사용한다.

In [2]:
aisles = pd.read_csv(DATA_DIR / "aisles.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

products_lookup = products[["product_id", "aisle_id", "department_id"]].copy()

department_mark_map = departments.sort_values("department_id").reset_index(drop=True).copy()
department_mark_map["mark"] = np.arange(len(department_mark_map), dtype=np.int16)

dept_to_mark = dict(zip(department_mark_map["department_id"], department_mark_map["mark"]))
dept_to_name = dict(zip(departments["department_id"], departments["department"]))

display(departments.head())
display(products.head())
display(department_mark_map.head())

print("products:", len(products))
print("aisles:", len(aisles))
print("departments:", len(departments))
print("department mark range:", department_mark_map["mark"].min(), department_mark_map["mark"].max())

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


,department_id,department,mark
0,1,frozen,0
1,2,other,1
2,3,bakery,2
3,4,produce,3
4,5,alcohol,4


products: 49688
aisles: 134
departments: 21
department mark range: 0 20


## 3. Reconstruct user-relative time

Instacart는 절대 날짜를 제공하지 않고, 유저별 이전 주문 후 경과일(`days_since_prior_order`)만 제공한다.

따라서 다음 상대 시간축을 만든다.

- `elapsed_days`: 유저별 `days_since_prior_order` 누적합
- `elapsed_days_hour`: `elapsed_days + order_hour_of_day / 24`
- `day_bucket`: `floor(elapsed_days)`

최종 quantity dataset은 `day_bucket` 기준 active bucket만 사용한다. 이렇게 하면 같은 날 여러 주문이 있어도 같은 bucket으로 합쳐져 `delta_t=0` 반복 문제가 줄어든다.

In [3]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
orders = orders.sort_values(["user_id", "order_number"]).reset_index(drop=True)

orders["elapsed_days"] = orders.groupby("user_id", sort=False)["days_since_prior_order"].transform(
    lambda s: s.fillna(0).cumsum()
)
orders["elapsed_days_hour"] = orders["elapsed_days"] + orders["order_hour_of_day"] / 24.0
orders["day_bucket"] = np.floor(orders["elapsed_days"]).astype("int32")

print("orders rows:", len(orders))
print("users:", orders["user_id"].nunique())
print("eval_set counts:")
display(orders["eval_set"].value_counts())

user_order_counts = orders.groupby("user_id", sort=False)["order_number"].max()
display(user_order_counts.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

time_diff = orders.groupby("user_id", sort=False)["elapsed_days_hour"].diff()
print("negative elapsed_days_hour diffs:", int((time_diff < 0).sum()))
print("non-positive elapsed_days_hour diffs:", int((time_diff <= 0).sum()))
print("same-day order transitions using elapsed_days only:", int((orders.groupby("user_id", sort=False)["elapsed_days"].diff() == 0).sum()))

orders rows: 3421083
users: 206209
eval_set counts:


eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

count    206209.000000
mean         16.590367
std          16.654774
min           4.000000
25%           6.000000
50%          10.000000
75%          20.000000
90%          38.000000
95%          52.000000
99%          89.000000
max         100.000000
Name: order_number, dtype: float64

negative elapsed_days_hour diffs: 0
non-positive elapsed_days_hour diffs: 21023
same-day order transitions using elapsed_days only: 67755


## 4. Aggregate product rows to order-level counts

`order_products__prior.csv`와 `order_products__train.csv`만 상품 label이 있다. `orders.csv`의 `test` 주문은 product row가 없으므로 quantity target 생성에서 제외한다.

여기서는 대용량 CSV를 chunk 단위로 읽어서 다음을 만든다.

- `basket_size`: 주문에 담긴 상품 row 수
- `reordered_count`: 주문 내 `reordered=1` row 수
- `dominant_department_id`: 주문 내 가장 많이 등장한 department

동률이면 더 작은 `department_id`를 선택해 deterministic하게 만든다.

In [4]:
def build_order_level_tables(
    data_dir: Path,
    products_lookup: pd.DataFrame,
    order_product_files: list[str],
    chunksize: int = 5_000_000,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    order_stat_parts: list[pd.DataFrame] = []
    dept_count_parts: list[pd.DataFrame] = []

    for file_name in order_product_files:
        path = data_dir / file_name
        print(f"reading {path.name}")

        reader = pd.read_csv(
            path,
            usecols=["order_id", "product_id", "reordered"],
            chunksize=chunksize,
        )

        for chunk_idx, chunk in enumerate(reader, start=1):
            chunk = chunk.merge(products_lookup, on="product_id", how="left", validate="many_to_one")
            if chunk["department_id"].isna().any():
                missing = chunk.loc[chunk["department_id"].isna(), "product_id"].drop_duplicates().head(10).tolist()
                raise ValueError(f"product_id not found in products.csv: {missing}")

            chunk["department_id"] = chunk["department_id"].astype("int16")

            order_stats = (
                chunk.groupby("order_id", sort=False)
                .agg(
                    basket_size=("product_id", "size"),
                    reordered_count=("reordered", "sum"),
                )
                .reset_index()
            )

            dept_counts = (
                chunk.groupby(["order_id", "department_id"], sort=False)
                .size()
                .reset_index(name="dept_count")
            )

            order_stat_parts.append(order_stats)
            dept_count_parts.append(dept_counts)

            if chunk_idx % LOG_EVERY_N_CHUNKS == 0:
                print(
                    f"  chunk {chunk_idx}: product rows={len(chunk):,}, "
                    f"orders={order_stats['order_id'].nunique():,}, "
                    f"order-departments={len(dept_counts):,}"
                )

            del chunk, order_stats, dept_counts
            gc.collect()

    order_stats_all = (
        pd.concat(order_stat_parts, ignore_index=True)
        .groupby("order_id", as_index=False, sort=False)[["basket_size", "reordered_count"]]
        .sum()
    )

    dept_counts_all = (
        pd.concat(dept_count_parts, ignore_index=True)
        .groupby(["order_id", "department_id"], as_index=False, sort=False)["dept_count"]
        .sum()
    )

    dominant_departments = (
        dept_counts_all.sort_values(["order_id", "dept_count", "department_id"], ascending=[True, False, True])
                    .drop_duplicates("order_id")
                    .rename(columns={"department_id": "dominant_department_id", "dept_count": "dominant_department_count"})
    )

    order_summary = order_stats_all.merge(dominant_departments, on="order_id", how="inner", validate="one_to_one")

    order_summary["basket_size"] = order_summary["basket_size"].astype("int16")
    order_summary["reordered_count"] = order_summary["reordered_count"].astype("int16")
    order_summary["dominant_department_id"] = order_summary["dominant_department_id"].astype("int16")
    order_summary["dominant_department_count"] = order_summary["dominant_department_count"].astype("int16")

    return order_summary, dept_counts_all


order_summary, order_department_counts = build_order_level_tables(
    data_dir=DATA_DIR,
    products_lookup=products_lookup,
    order_product_files=ORDER_PRODUCT_FILES,
    chunksize=CHUNKSIZE,
)

print("order_summary rows:", len(order_summary))
print("order_department_counts rows:", len(order_department_counts))
display(order_summary.head())
display(order_summary["basket_size"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

reading order_products__prior.csv
  chunk 1: product rows=5,000,000, orders=495,727, order-departments=2,346,614
  chunk 2: product rows=5,000,000, orders=495,988, order-departments=2,347,848
  chunk 3: product rows=5,000,000, orders=495,130, order-departments=2,347,763
  chunk 4: product rows=5,000,000, orders=495,142, order-departments=2,346,386
  chunk 5: product rows=5,000,000, orders=495,628, order-departments=2,347,138
  chunk 6: product rows=5,000,000, orders=495,872, order-departments=2,346,930
  chunk 7: product rows=2,434,489, orders=241,393, order-departments=1,143,534
reading order_products__train.csv
  chunk 1: product rows=1,384,617, orders=131,209, order-departments=640,391
order_summary rows: 3346083
order_department_counts rows: 15866589


,order_id,basket_size,reordered_count,dominant_department_id,dominant_department_count
0,2,9,6,13,5
1,3,8,8,4,3
2,4,13,12,14,4
3,5,26,21,4,7
4,6,3,0,17,2


count    3.346083e+06
mean     1.010707e+01
std      7.542326e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.400000e+01
90%      2.000000e+01
95%      2.500000e+01
99%      3.500000e+01
max      1.450000e+02
Name: basket_size, dtype: float64

## 5. Build active-day quantity reconstruction table

이 단계에서 주문 단위 정보를 `user_id + day_bucket` 단위로 묶는다.

- `demand_qty`: 해당 active day의 전체 상품 row 수
- `delta_t`: 같은 user의 이전 active day와의 day 차이
- `z`: `log1p(demand_qty)`
- `mark`: dominant department를 `0..20`으로 remap한 값

`marked_target_df.parquet`와 호환되는 7개 컬럼도 함께 만든다.

In [5]:
observed_orders = orders.merge(order_summary, on="order_id", how="inner", validate="one_to_one")
observed_orders = observed_orders.sort_values(["user_id", "day_bucket", "order_number"]).reset_index(drop=True)

print("observed orders with product labels:", len(observed_orders))
print("observed users:", observed_orders["user_id"].nunique())
display(observed_orders["eval_set"].value_counts())

order_time = observed_orders[["order_id", "user_id", "day_bucket"]].copy()

daily_department_counts = order_department_counts.merge(order_time, on="order_id", how="inner", validate="many_to_one")
daily_department_counts = (
    daily_department_counts.groupby(["user_id", "day_bucket", "department_id"], as_index=False, sort=False)["dept_count"]
    .sum()
)

daily_dominant_department = (
    daily_department_counts.sort_values(
        ["user_id", "day_bucket", "dept_count", "department_id"],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(["user_id", "day_bucket"])
    .rename(columns={"department_id": "dominant_department_id", "dept_count": "dominant_department_count"})
)

daily = (
    observed_orders.groupby(["user_id", "day_bucket"], as_index=False, sort=False)
    .agg(
        demand_qty=("basket_size", "sum"),
        order_count=("order_id", "size"),
        reordered_count=("reordered_count", "sum"),
        first_order_number=("order_number", "first"),
        last_order_number=("order_number", "last"),
        first_order_hour=("order_hour_of_day", "first"),
        last_order_hour=("order_hour_of_day", "last"),
        first_elapsed_days_hour=("elapsed_days_hour", "first"),
        last_elapsed_days_hour=("elapsed_days_hour", "last"),
        eval_set_latest=("eval_set", "last"),
    )
)

daily = daily.merge(
    daily_dominant_department[["user_id", "day_bucket", "dominant_department_id", "dominant_department_count"]],
    on=["user_id", "day_bucket"],
    how="inner",
    validate="one_to_one",
)

daily = daily.sort_values(["user_id", "day_bucket"]).reset_index(drop=True)
daily["delta_t"] = daily.groupby("user_id", sort=False)["day_bucket"].diff().fillna(0).astype("int16")
daily["demand_qty"] = daily["demand_qty"].astype("float64")
daily["z"] = np.log1p(daily["demand_qty"])
daily["mark"] = daily["dominant_department_id"].map(dept_to_mark).astype("int16")
daily["dominant_department"] = daily["dominant_department_id"].map(dept_to_name)
daily["reorder_ratio"] = daily["reordered_count"] / daily["demand_qty"]

daily["oper_part_no"] = "user_" + daily["user_id"].astype(str)
daily["demand_dt"] = daily["day_bucket"].astype("int32")
daily["seq"] = daily["day_bucket"].astype("int32")

marked_target_like = daily[["oper_part_no", "demand_dt", "seq", "delta_t", "demand_qty", "z", "mark"]].copy()

print("daily active rows:", len(daily))
print("daily users:", daily["user_id"].nunique())
display(marked_target_like.head())
display(marked_target_like["demand_qty"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
display(marked_target_like["delta_t"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

observed orders with product labels: 3346083
observed users: 206209


eval_set
prior    3214874
train     131209
Name: count, dtype: int64

daily active rows: 3279521
daily users: 206209


,oper_part_no,demand_dt,seq,delta_t,demand_qty,z,mark
0,user_1,0,0,0,5.0,1.791759,18
1,user_1,15,15,15,6.0,1.945910,18
2,user_1,36,36,21,5.0,1.791759,18
3,user_1,65,65,29,5.0,1.791759,18
4,user_1,93,93,28,8.0,2.197225,3


count    3.279521e+06
mean     1.031221e+01
std      7.724334e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.400000e+01
90%      2.000000e+01
95%      2.500000e+01
99%      3.600000e+01
max      1.770000e+02
Name: demand_qty, dtype: float64

count    3.279521e+06
mean     1.050507e+01
std      9.196010e+00
min      0.000000e+00
25%      4.000000e+00
50%      7.000000e+00
75%      1.500000e+01
90%      3.000000e+01
95%      3.000000e+01
99%      3.000000e+01
max      3.000000e+01
Name: delta_t, dtype: float64

## 6. Quality checks

생성한 table이 TPP/quantity reconstruction 실험에 들어가기 전에 기본 품질을 확인한다.

In [6]:
def assert_quantity_table_quality(df: pd.DataFrame, mark_map: pd.DataFrame) -> None:
    duplicate_keys = df.duplicated(["oper_part_no", "seq"]).sum()
    assert duplicate_keys == 0, f"duplicated entity-time rows: {duplicate_keys}"

    null_counts = df.isna().sum()
    assert int(null_counts.sum()) == 0, f"null values found:\n{null_counts[null_counts > 0]}"

    assert (df["delta_t"] >= 0).all(), "negative delta_t exists"
    assert (df["demand_qty"] > 0).all(), "non-positive demand_qty exists"
    assert np.isfinite(df["z"]).all(), "non-finite z exists"

    min_mark = int(df["mark"].min())
    max_mark = int(df["mark"].max())
    assert min_mark >= 0, f"invalid min mark: {min_mark}"
    assert max_mark < len(mark_map), f"invalid max mark: {max_mark}"

    first_delta = df.sort_values(["oper_part_no", "seq"]).groupby("oper_part_no", sort=False)["delta_t"].first()
    assert (first_delta == 0).all(), "first delta_t must be 0 for every entity"


assert_quantity_table_quality(marked_target_like, department_mark_map)

print("quality checks passed")
print("shape:", marked_target_like.shape)
print("entities:", marked_target_like["oper_part_no"].nunique())
print("mark counts:")
display(marked_target_like["mark"].value_counts().sort_index().to_frame("count"))

entity_lengths = marked_target_like.groupby("oper_part_no", sort=False).size()
print("entity length distribution:")
display(entity_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

quality checks passed
shape: (3279521, 7)
entities: 206209
mark counts:


,count
mark,
0,275851
1,3233
2,80457
3,1613419
4,29640
5,7998
6,298250
7,8910
8,27930


entity length distribution:


count    206209.000000
mean         15.903869
std          16.044227
min           1.000000
25%           6.000000
50%          10.000000
75%          20.000000
90%          37.000000
95%          50.000000
99%          83.000000
max         100.000000
dtype: float64

## 7. Chronological split

기본 출력에는 전체 table을 저장한다. 추가로 유저별 chronological split column을 만든다.

- event 수가 5개 이상인 user: 마지막 event는 `test`, 마지막 직전 event는 `validation`, 나머지는 `train`
- event 수가 4개인 user: 마지막 event는 `validation`, 나머지는 `train`
- event 수가 3개 이하인 user: 모두 `train`

실험 코드에서 별도 split을 쓴다면 이 column은 무시해도 된다.

In [7]:
daily = daily.sort_values(["user_id", "seq"]).reset_index(drop=True)
daily["event_idx"] = daily.groupby("user_id", sort=False).cumcount()
daily["n_events"] = daily.groupby("user_id", sort=False)["seq"].transform("size")
daily["chronological_split"] = "train"

validation_mask = ((daily["n_events"] >= 5) & (daily["event_idx"] == daily["n_events"] - 2)) | (
    (daily["n_events"] == 4) & (daily["event_idx"] == daily["n_events"] - 1)
)
test_mask = (daily["n_events"] >= 5) & (daily["event_idx"] == daily["n_events"] - 1)

daily.loc[validation_mask, "chronological_split"] = "validation"
daily.loc[test_mask, "chronological_split"] = "test"

marked_target_like_with_split = daily[
    [
        "oper_part_no",
        "demand_dt",
        "seq",
        "delta_t",
        "demand_qty",
        "z",
        "mark",
        "chronological_split",
    ]
].copy()

display(marked_target_like_with_split["chronological_split"].value_counts())
display(marked_target_like_with_split.head())

chronological_split
train         2908809
validation     196615
test           174097
Name: count, dtype: int64

,oper_part_no,demand_dt,seq,delta_t,demand_qty,z,mark,chronological_split
0,user_1,0,0,0,5.0,1.791759,18,train
1,user_1,15,15,15,6.0,1.945910,18,train
2,user_1,36,36,21,5.0,1.791759,18,train
3,user_1,65,65,29,5.0,1.791759,18,train
4,user_1,93,93,28,8.0,2.197225,3,train


## 8. Save outputs

저장 파일은 세 종류다.

- `instacart_marked_target_df.parquet`: 기존 `marked_target_df`와 호환되는 7개 컬럼
- `instacart_marked_target_with_split.parquet`: split column 포함
- `instacart_quantity_reconstruction_rich.parquet`: 진단/해석용 보조 컬럼 포함
- `instacart_department_mark_map.csv`: mark와 원래 department id/name 매핑

Parquet 엔진이 없는 환경이면 CSV fallback을 저장한다.

In [9]:
def save_table(df: pd.DataFrame, output_dir: Path, stem: str) -> Path:
    parquet_path = output_dir / f"{stem}.parquet"
    csv_path = output_dir / f"{stem}.csv"

    try:
        df.to_parquet(parquet_path, index=False)
        print(f"saved parquet: {parquet_path}")
        return parquet_path
    except Exception as exc:
        print(f"parquet save failed for {stem}: {exc}")
        df.to_csv(csv_path, index=False)
        print(f"saved csv fallback: {csv_path}")
        return csv_path


rich_columns = [
    "user_id",
    "oper_part_no",
    "demand_dt",
    "seq",
    "delta_t",
    "demand_qty",
    "z",
    "mark",
    "dominant_department_id",
    "dominant_department",
    "dominant_department_count",
    "order_count",
    "reordered_count",
    "reorder_ratio",
    "first_order_number",
    "last_order_number",
    "first_order_hour",
    "last_order_hour",
    "first_elapsed_days_hour",
    "last_elapsed_days_hour",
    "eval_set_latest",
    "event_idx",
    "n_events",
    "chronological_split",
]
rich_table = daily[rich_columns].copy()

saved_paths = []
saved_paths.append(save_table(marked_target_like, OUTPUT_DIR, "instacart_marked_target_df"))
saved_paths.append(save_table(marked_target_like_with_split, OUTPUT_DIR, "instacart_marked_target_with_split"))
saved_paths.append(save_table(rich_table, OUTPUT_DIR, "instacart_quantity_reconstruction_rich"))

mark_map_path = OUTPUT_DIR / "instacart_department_mark_map.csv"
department_mark_map.to_csv(mark_map_path, index=False)
saved_paths.append(mark_map_path)
print(f"saved mark map: {mark_map_path}")

print("\nSaved files:")
for path in saved_paths:
    print("-", path)

saved parquet: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_df.parquet
saved parquet: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_with_split.parquet
saved parquet: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_quantity_reconstruction_rich.parquet
saved mark map: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_department_mark_map.csv

Saved files:
- /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_df.parquet
- /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_with_split.parquet
- /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_quantity_reconstruction_rich.parquet
- /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_departm

## 9. Optional: user-department demand variant

위 기본 데이터셋은 `entity=user_id`, `mark=dominant_department`이다.

더 세밀한 변형으로 `entity=user_id + department_id`를 만들 수도 있다. 이 경우 `demand_qty`는 해당 유저가 해당 department에서 구매한 상품 row 수가 된다. 다만 entity 수가 크게 늘고 sequence가 더 sparse해지므로 첫 실험에는 기본 데이터셋을 권장한다.

아래 셀은 필요할 때만 `BUILD_USER_DEPARTMENT_VARIANT=True`로 바꿔 실행한다.

In [10]:
BUILD_USER_DEPARTMENT_VARIANT = False

if BUILD_USER_DEPARTMENT_VARIANT:
    user_dept_daily = daily_department_counts.sort_values(["user_id", "department_id", "day_bucket"]).copy()
    user_dept_daily["oper_part_no"] = (
        "user_" + user_dept_daily["user_id"].astype(str) + "_dept_" + user_dept_daily["department_id"].astype(str)
    )
    user_dept_daily["demand_dt"] = user_dept_daily["day_bucket"].astype("int32")
    user_dept_daily["seq"] = user_dept_daily["day_bucket"].astype("int32")
    user_dept_daily["delta_t"] = (
        user_dept_daily.groupby("oper_part_no", sort=False)["seq"].diff().fillna(0).astype("int16")
    )
    user_dept_daily["demand_qty"] = user_dept_daily["dept_count"].astype("float64")
    user_dept_daily["z"] = np.log1p(user_dept_daily["demand_qty"])
    user_dept_daily["mark"] = user_dept_daily["department_id"].map(dept_to_mark).astype("int16")

    user_dept_marked = user_dept_daily[
        ["oper_part_no", "demand_dt", "seq", "delta_t", "demand_qty", "z", "mark"]
    ].copy()
    assert_quantity_table_quality(user_dept_marked, department_mark_map)
    save_table(user_dept_marked, OUTPUT_DIR, "instacart_user_department_marked_target_df")

    print("user-department rows:", len(user_dept_marked))
    print("user-department entities:", user_dept_marked["oper_part_no"].nunique())
    display(user_dept_marked.head())
else:
    print("Skipped user-department variant. Set BUILD_USER_DEPARTMENT_VARIANT=True to build it.")

Skipped user-department variant. Set BUILD_USER_DEPARTMENT_VARIANT=True to build it.


## 10. Interpretation notes

이 데이터셋을 논문/실험에 포함할 때는 다음 표현이 적절하다.

- Instacart 원자료는 유저별 주문 sequence와 주문 내 상품 row를 제공한다.
- 절대 timestamp는 없기 때문에 calendar-time demand가 아니라 user-relative time demand로 재구성했다.
- quantity는 원래 명시된 판매 수량이 아니라 상품 row count에서 파생한 derived basket-count demand이다.
- mark는 해당 active day basket의 dominant department이다.

따라서 이 데이터는 `yellow_trip`의 pickup-count 방식과 비슷한 count reconstruction 문제지만, 시간축 해석은 더 제한적이다.